In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_kopi = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df_kopi.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 08:18:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 08:18:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx").fit(df_kopi)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_kopi)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df_kopi)

df_kopi_idx = indexer_menu.transform(df_kopi)
df_kopi_idx = indexer_cabang.transform(df_kopi_idx)
df_kopi_idx = indexer_waktu.transform(df_kopi_idx)

assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_kopi_final = assembler_utama.transform(df_kopi_idx)
df_kopi_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [3]:
from pyspark.ml.regression import LinearRegression

train_kopi, test_kopi = df_kopi_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train_kopi.count()}")
print(f"Jumlah data testing   : {test_kopi.count()}")

lr_kopi = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_kopi = lr_kopi.fit(train_kopi)

print("Koefisien fitur:", model_kopi.coefficients)
print("Intercept:", model_kopi.intercept)

Jumlah data training : 1839
Jumlah data testing   : 661
Koefisien fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

pred_kopi = model_kopi.transform(test_kopi)

rmse_kopi = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="rmse"
).evaluate(pred_kopi)

r2_kopi = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
).evaluate(pred_kopi)

print(f"RMSE Model Utama : Rp {rmse_kopi:,.0f}")
print(f"R² Model Utama   : {r2_kopi:.4f} ({r2_kopi*100:.1f}%)")

RMSE Model Utama : Rp 10,901
R² Model Utama   : 0.9107 (91.1%)


In [5]:
assembler_dua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)

df_kopi_dua = assembler_dua.transform(df_kopi_idx)
train_2, test_2 = df_kopi_dua.randomSplit([0.75, 0.25], seed=101)

lr_dua = LinearRegression(featuresCol="features_2", labelCol="total_penjualan")
model_dua = lr_dua.fit(train_2)

pred_dua = model_dua.transform(test_2)
r2_dua = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
).evaluate(pred_dua)

print(f"R² Model 5 Fitur (Lengkap)  : {r2_kopi:.4f}")
print(f"R² Model 2 Fitur (Sederhana): {r2_dua:.4f}")

R² Model 5 Fitur (Lengkap)  : 0.9107
R² Model 2 Fitur (Sederhana): 0.9108


Kesimpulan Eksperimen:

Menambahkan informasi nama_menu, cabang, dan waktu_pesan tidak meningkatkan akurasi prediksi secara berarti (nilai R² model 2 fitur hampir identik atau bahkan sedikit lebih tinggi daripada model 5 fitur). Hal ini terjadi karena pada dataset simulasi, total_penjualan murni dibentuk dari perkalian qty * harga_satuan ditambah noise acak, sehingga variabel menu, cabang, dan waktu tidak memiliki pengaruh matematis langsung terhadap nilai total transaksi.

In [6]:
data_baru = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Cappuccino"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Sore", "Malam"],
    "qty": [2, 4, 1],
    "harga_satuan": [20000, 25000, 30000]
})

df_baru = spark.createDataFrame(data_baru)

df_baru_idx = indexer_menu.transform(df_baru)
df_baru_idx = indexer_cabang.transform(df_baru_idx)
df_baru_idx = indexer_waktu.transform(df_baru_idx)
df_baru_final = assembler_utama.transform(df_baru_idx)

hasil_prediksi_baru = model_kopi.transform(df_baru_final)
hasil_prediksi_baru.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |37525.35198441948|
|Roti Bakar|Yogyakarta|Sore       |4  |25000       |97661.84100990095|
|Cappuccino|Semarang  |Malam      |1  |30000       |46482.8909835706 |
+----------+----------+-----------+---+------------+-----------------+



In [7]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
